In [ ]:
import json, pandas as pd, numpy as np, os
from pathlib import Path

with open('data/raw/finagent_config.json') as f:
    cfg = json.load(f)
TICKERS     = cfg['TICKERS']
MACRO       = cfg['MACRO']
ALL_SYMBOLS = cfg['ALL_SYMBOLS']
PERIOD      = cfg['PERIOD']
GROQ_API_KEY = cfg['GROQ_API_KEY']
FRED_API_KEY = cfg['FRED_API_KEY']

os.environ['GROQ_API_KEY'] = GROQ_API_KEY

def load_csv(path, index_col='Date'):
    p = Path(path)
    if p.exists(): return pd.read_csv(p, index_col=index_col, parse_dates=True)
    print(f'  ⚠️  Missing: {path}'); return None

clean_dfs = {}
for symbol in ALL_SYMBOLS:
    safe = symbol.replace('=','_')
    df = load_csv(f'data/processed/{safe}_clean.csv')
    if df is not None: clean_dfs[symbol] = df

fund_df  = load_csv('data/processed/fundamentals.csv', index_col=0)
macro_df = load_csv('data/processed/macro_fred.csv')

print(f'✅ Config + {len(clean_dfs)} datasets loaded')
print(f'Fundamentals: {"✅" if fund_df is not None else "⚠️  missing"}')
print(f'Macro FRED  : {"✅" if macro_df is not None else "⚠️  missing"}')

✅ Config + 6 datasets loaded
Fundamentals: ✅
Macro FRED  : ✅


C:\Users\VIP\AppData\Local\Temp\ipykernel_12388\2285306182.py:17: UserWarning: Could not infer format, so each element will be parsed individually, falling back to `dateutil`. To ensure parsing is consistent and as-expected, please specify a format.
  if p.exists(): return pd.read_csv(p, index_col=index_col, parse_dates=True)


Setup Groq (Llama 3.3 70B)

In [3]:
import subprocess, sys, time
from groq import Groq

In [4]:
_groq_client = Groq(api_key=GROQ_API_KEY)

def call_llm(prompt: str, system: str = '', max_tokens: int = 1024) -> str:
    """Call Groq Llama 3.3 70B — free, works in VN."""
    if not GROQ_API_KEY or GROQ_API_KEY.startswith('gsk_xxx'):
        return '[DEMO MODE — set GROQ_API_KEY in Module 1 config]'
    try:
        resp = _groq_client.chat.completions.create(
            model='llama-3.3-70b-versatile',
            max_tokens=max_tokens,
            messages=[
                {'role':'system', 'content': system or 'You are a professional financial analyst. Be concise and precise.'},
                {'role':'user',   'content': prompt}
            ]
        )
        time.sleep(1)
        return resp.choices[0].message.content
    except Exception as e:
        return f'[GROQ ERROR] {str(e)}'

# Quick test
test = call_llm('Say exactly: Groq OK', max_tokens=20)
print('✅ Groq API:', test)

✅ Groq API: Groq OK


# Market summary

In [5]:
def build_market_summary(dfs, tickers):
    summary = {}
    for t in tickers:
        if t not in dfs: continue
        df   = dfs[t].dropna(subset=['Close','Daily_Return'])
        last = df.iloc[-1]
        summary[t] = {
            'current_price'    : round(float(last['Close']), 2),
            'period_return_pct': round(float((df['Close'].iloc[-1]/df['Close'].iloc[0]-1)*100), 2),
            'mean_daily_ret'   : round(float(df['Daily_Return'].mean()*100), 4),
            'volatility_ann'   : round(float(last.get('Volatility_30', float('nan'))), 4),
            'rsi_14'           : round(float(last.get('RSI_14', float('nan'))), 1),
            'n_outlier_days'   : int(df.get('Outlier_Flag', pd.Series(False)).sum()),
            'date_range'       : f"{df.index[0].date()} to {df.index[-1].date()}"
        }
    return summary

mkt = build_market_summary(clean_dfs, TICKERS)
print('📊 Market summary:')
for k,v in mkt.items():
    print(f'  {k:6} price=${v["current_price"]:>10,.2f}  return={v["period_return_pct"]:>7.2f}%  RSI={v["rsi_14"]}')

📊 Market summary:
  AAPL   price=$    298.97  return= 110.28%  RSI=69.8
  MSFT   price=$    417.42  return=  52.33%  RSI=49.8
  NVDA   price=$    220.61  return=1033.98%  RSI=72.6
  GOOG   price=$    384.90  return= 186.98%  RSI=75.8


# Trend summary

In [6]:
trend_prompt = f"""You are a quantitative financial analyst writing a market brief.

Here is the summary data for {len(TICKERS)} stocks over the past {PERIOD}:
{json.dumps(mkt, indent=2)}

Write a 200-word market trend summary covering:
1. Overall market direction and momentum
2. Top performer and laggard with reasoning
3. RSI signals — any overbought (>70) or oversold (<30) conditions?
4. Volatility observations

Use professional financial language. Be specific with numbers."""

trend_report = call_llm(trend_prompt)
print('=== AI TREND SUMMARY ===')
print(trend_report)

=== AI TREND SUMMARY ===
**Market Trend Summary**

The overall market direction exhibits a bullish trend, with the top performer, NVDA, yielding a 1033.98% return over the past 5 years. This outperformance is attributed to its remarkably high mean daily return of 5.1933%. In contrast, MSFT lags behind, with a 52.33% return, likely due to its relatively lower mean daily return of 1.0273%.

From a technical perspective, the Relative Strength Index (RSI) signals overbought conditions for NVDA (72.6), GOOG (75.8), and AAPL (69.8), indicating potential consolidation or pullback. No oversold conditions are observed.

Volatility observations reveal a range of 0.8994 (AAPL) to 1.6801 (NVDA), with GOOG and MSFT exhibiting volatilities of 1.4206 and 0.9967, respectively. The notable disparity in volatility between NVDA and the other stocks may be a function of its superior growth momentum. Overall, the market trend suggests continued upward momentum, albeit with potential for near-term correctio

# Anomoly detection

In [7]:
anomaly_data = {}
for t in TICKERS:
    if t not in clean_dfs: continue
    df = clean_dfs[t]
    flagged = df[df.get('Outlier_Flag', pd.Series(False, index=df.index))]
    if len(flagged):
        anomaly_data[t] = [
            {'date':str(idx.date()), 'close':round(float(row['Close']),2),
             'daily_return_pct':round(float(row['Daily_Return'])*100,2)}
            for idx, row in flagged.iterrows()
        ]

anomaly_prompt = f"""Analyse flagged anomalous trading days for {TICKERS}.
These days exceeded 3× IQR or |return| > 40%.

Anomaly data:
{json.dumps(anomaly_data if anomaly_data else {'note':'No outliers detected — data appears clean'}, indent=2)}

For each anomaly, speculate on probable causes (earnings, macro event, market crash, data error).
Conclude with a brief assessment of data quality and reliability."""

anomaly_report = call_llm(anomaly_prompt)
print('=== AI ANOMALY DETECTION ===')
print(anomaly_report)

=== AI ANOMALY DETECTION ===
Analysis:

Given the anomaly data states "No outliers detected — data appears clean", there are no flagged anomalous trading days for ['AAPL', 'MSFT', 'NVDA', 'GOOG'] that exceeded 3× IQR or |return| > 40%.

Speculation:

* No speculation is required as there are no anomalies to analyze.

Assessment:

Data quality and reliability appear to be high, as no outliers or anomalies were detected in the trading data for the specified stocks. This suggests that the data is clean and free from errors or unusual patterns that could impact analysis or decision-making.


# Risk Commentary & Asset Comparison

In [8]:
risk_prompt = f"""Compare these assets from a risk-adjusted return perspective:

{json.dumps(mkt, indent=2)}

Your analysis must include:
1. Risk-return ranking (use annualised volatility and period return; assume 0% risk-free rate)
2. Diversification assessment: do assets move together or independently?
3. Which asset suits a conservative investor vs an aggressive one, and why?
4. Specific risk warnings (e.g. high RSI, high volatility, negative return trend)

Format as a structured analyst report with clear headings."""

risk_report = call_llm(risk_prompt, max_tokens=1500)
print('=== AI RISK COMMENTARY & COMPARISON ===')
print(risk_report)

=== AI RISK COMMENTARY & COMPARISON ===
**Risk-Adjusted Return Analysis Report**

### 1. Risk-Return Ranking

To evaluate the assets from a risk-adjusted return perspective, we calculated the Sharpe Ratio for each asset, assuming a 0% risk-free rate. The Sharpe Ratio is defined as the period return percentage divided by the annualized volatility.

| Asset | Period Return (%) | Annualized Volatility | Sharpe Ratio |
| --- | --- | --- | --- |
| NVDA | 1033.98 | 1.6801 | 6.15 |
| GOOG | 186.98 | 1.4206 | 1.32 |
| AAPL | 110.28 | 0.8994 | 1.23 |
| MSFT | 52.33 | 0.9967 | 0.52 |

Based on the Sharpe Ratio, the risk-return ranking is:
1. NVDA
2. GOOG
3. AAPL
4. MSFT

### 2. Diversification Assessment

To assess the diversification potential of these assets, we examined their mean daily returns and volatility. Assets with low correlations and different return profiles can provide diversification benefits.

* NVDA has the highest mean daily return (5.1933) and volatility (1.6801), indicating a

# Fundamental Valuation

In [9]:
if fund_df is None:
    fundamental_report = '[Fundamental data not available — run Module 2 first]'
else:
    fund_summary = {}
    for t in TICKERS:
        if t not in fund_df.index: continue
        fund_summary[t] = {
            'P/E (trailing)': fund_df.loc[t,'P/E Ratio'], 'P/E (forward)': fund_df.loc[t,'Forward P/E'],
            'EPS (TTM)': fund_df.loc[t,'EPS (TTM)'], 'Total Revenue': fund_df.loc[t,'Total Revenue ($)'],
            'Profit Margin': fund_df.loc[t,'Profit Margin'], 'Gross Margin': fund_df.loc[t,'Gross Margin'],
            'Debt/Equity': fund_df.loc[t,'Debt/Equity'], 'ROE': fund_df.loc[t,'ROE'],
            'Beta': fund_df.loc[t,'Beta'], 'Market Cap': fund_df.loc[t,'Market Cap ($)'],
        }
    fundamental_prompt = f"""You are a professional equity analyst. Analyse fundamental data for {TICKERS}:

{json.dumps(fund_summary, indent=2)}

Cover:
1. VALUATION: Is each stock cheap or expensive? Which offers best value?
2. PROFITABILITY: Compare margins. Which converts revenue to profit most efficiently?
3. EARNINGS QUALITY: Comment on EPS levels.
4. RISK: Highest financial risk based on Debt/Equity and Beta?
5. VERDICT: Rank stocks from most to least attractive. One-line reason each.

Be specific with numbers. Use professional financial language."""
    fundamental_report = call_llm(fundamental_prompt, max_tokens=1500)

print('=== AI FUNDAMENTAL VALUATION ===')
print(fundamental_report)

=== AI FUNDAMENTAL VALUATION ===
**VALUATION**: Based on the P/E ratios, NVDA is the most expensive stock with a trailing P/E of 45.02, while MSFT appears to offer the best value with a forward P/E of 21.55, implying a relatively lower valuation. 

**PROFITABILITY**: NVDA has the highest profit margin at 55.60% and gross margin at 71.07%, indicating it converts revenue to profit most efficiently. MSFT also demonstrates strong margins, with a profit margin of 39.34% and a gross margin of 68.31%.

**EARNINGS QUALITY**: EPS levels are relatively high across the board, with MSFT boasting the highest EPS (TTM) at $16.77. NVDA's EPS (TTM) is the lowest at $4.9, which may be a concern despite its high margins.

**RISK**: NVDA has the highest beta at 2.24, indicating higher market risk. However, its Debt/Equity ratio is relatively low at 7.25. AAPL has the highest Debt/Equity ratio at 79.55, posing significant financial risk, especially when combined with its beta of 1.06.

**VERDICT**: Ranked

# Macro Environment

In [10]:
if macro_df is None:
    macro_report = '[Macro data not available — run Module 2 first]'
else:
    latest  = macro_df.iloc[-1].to_dict()
    prev_6m = macro_df.iloc[-7].to_dict() if len(macro_df)>6 else macro_df.iloc[0].to_dict()
    macro_snapshot = {
        'current': {k:round(v,2) for k,v in latest.items()},
        '6mo_ago': {k:round(v,2) for k,v in prev_6m.items()},
        'change' : {k:round(latest.get(k,0)-prev_6m.get(k,0),2) for k in latest},
        'stocks_analysed': TICKERS, 'stock_period': PERIOD,
    }
    macro_prompt = f"""You are a macro-economic analyst covering US equity markets.

Macroeconomic data:
{json.dumps(macro_snapshot, indent=2)}

Portfolio: {', '.join(TICKERS)}

Write 300 words covering:
1. MONETARY POLICY: Fed Funds Rate direction and equity valuation implications
2. INFLATION: CPI vs 2% target; which sectors benefit/suffer?
3. TREASURY YIELD: P/E compression/expansion risk; yield curve inversion?
4. LABOUR MARKET: Unemployment and recession risk
5. PORTFOLIO IMPACT: Which tickers are most/least exposed to macro risk?

Conclude: MACRO RISK RATING: Low / Medium / High + one-sentence justification."""
    macro_report = call_llm(macro_prompt, max_tokens=1500)

print('=== AI MACRO ENVIRONMENT ===')
print(macro_report)

=== AI MACRO ENVIRONMENT ===
Based on the provided macroeconomic data, we can assess the potential impact on the US equity market and the given portfolio.

1. **MONETARY POLICY**: The lack of current Fed Funds Rate data limits our analysis. However, the 6mo_ago rate of 3.88% is relatively high, suggesting a tighter monetary policy. If the rate increases further, it may lead to lower equity valuations, as higher borrowing costs can negatively impact corporate profits.
2. **INFLATION**: Without current CPI data, we cannot directly compare it to the 2% target. However, the 6mo_ago rate of 2.99% is close to the target. Sectors that typically benefit from moderate inflation include consumer staples and healthcare, while those that may suffer include technology and finance.
3. **TREASURY YIELD**: The 10Y Treasury Yield has increased by 0.59% over the past 6 months, which may lead to P/E compression in the equity market. A rising yield curve can also indicate a growing economy, but the curren

# Report

In [ ]:
from datetime import date
report_date = date.today().strftime('%Y-%m-%d')
Path('outputs/reports').mkdir(parents=True, exist_ok=True)

full_report = f"""# FinAgent AI Analysis Report
**Generated:** {report_date}  
**Tickers:** {', '.join(TICKERS)} | **Macro:** {', '.join(MACRO)} | **Period:** {PERIOD}

---

## 1. Market Trend Summary

{trend_report}

---

## 2. Anomaly Detection

{anomaly_report}

---

## 3. Risk Commentary & Asset Comparison

{risk_report}

---

## 4. Fundamental Valuation Analysis

{fundamental_report}

---

## 5. Macro Environment Analysis

{macro_report}

---
*Generated by FinAgent · Groq Llama 3.3 70B · Yahoo Finance / FRED 
"""

report_path = f'outputs/reports/ai_analysis_{report_date}.md'
with open(report_path, 'w', encoding='utf-8') as f:
    f.write(full_report)
print(f'✅ AI report saved: {report_path}')

✅ AI report saved: outputs/reports/ai_analysis_2026-05-21.md


PDF file export

In [ ]:
import re

from reportlab.platypus import (
    SimpleDocTemplate, Paragraph, Spacer, Table, TableStyle, HRFlowable
)
from reportlab.lib.styles import getSampleStyleSheet, ParagraphStyle
from reportlab.lib.pagesizes import letter
from reportlab.lib import colors
from reportlab.lib.units import inch
from reportlab.lib.enums import TA_LEFT, TA_CENTER

# ─────────────────────────────────────────────────────────────────────
# Helpers
# ─────────────────────────────────────────────────────────────────────

def build_styles():
    """Return a dict of all custom paragraph styles."""
    base = getSampleStyleSheet()

    custom = {}

    custom['Title'] = ParagraphStyle(
        'MyTitle',
        parent=base['Title'],
        fontSize=20,
        textColor=colors.HexColor('#1e3a5f'),
        spaceAfter=6,
        alignment=TA_CENTER,
    )
    custom['Subtitle'] = ParagraphStyle(
        'Subtitle',
        parent=base['Normal'],
        fontSize=10,
        textColor=colors.HexColor('#555555'),
        spaceAfter=4,
        alignment=TA_CENTER,
    )
    custom['SectionHeading'] = ParagraphStyle(
        'SectionHeading',
        parent=base['Heading2'],
        fontSize=13,
        textColor=colors.HexColor('#1e3a5f'),
        spaceBefore=14,
        spaceAfter=6,
        borderPad=2,
    )
    custom['SubHeading'] = ParagraphStyle(
        'SubHeading',
        parent=base['Heading3'],
        fontSize=11,
        textColor=colors.HexColor('#2c5282'),
        spaceBefore=8,
        spaceAfter=4,
        bold=True,
    )
    custom['Body'] = ParagraphStyle(
        'Body',
        parent=base['BodyText'],
        fontSize=9.5,
        leading=14,
        textColor=colors.HexColor('#222222'),
        spaceAfter=6,
    )
    custom['BulletBody'] = ParagraphStyle(
        'BulletBody',
        parent=base['BodyText'],
        fontSize=9.5,
        leading=14,
        textColor=colors.HexColor('#222222'),
        leftIndent=14,
        spaceAfter=3,
    )
    custom['TableCell'] = ParagraphStyle(
        'TableCell',
        parent=base['BodyText'],
        fontSize=8.5,
        leading=12,
    )
    custom['TableHeader'] = ParagraphStyle(
        'TableHeader',
        parent=base['BodyText'],
        fontSize=8.5,
        leading=12,
        textColor=colors.white,
        fontName='Helvetica-Bold',
    )
    custom['Footer'] = ParagraphStyle(
        'Footer',
        parent=base['Normal'],
        fontSize=8,
        textColor=colors.HexColor('#888888'),
        alignment=TA_CENTER,
        spaceAfter=0,
    )
    return custom


def escape_xml(text: str) -> str:
    """Escape special XML chars so ReportLab Paragraph doesn't choke."""
    text = text.replace('&', '&amp;')
    text = text.replace('<', '&lt;')
    text = text.replace('>', '&gt;')
    return text


def md_inline_to_xml(text: str) -> str:
    """
    Convert lightweight Markdown inline syntax to ReportLab XML tags.
    Call AFTER escape_xml so we don't double-escape the tag brackets.
    """
    # Bold: **text** → <b>text</b>
    text = re.sub(r'\*\*(.*?)\*\*', r'<b>\1</b>', text)
    # Italic: *text* → <i>text</i>  (single asterisk, not followed by another)
    text = re.sub(r'(?<!\*)\*(?!\*)(.*?)(?<!\*)\*(?!\*)', r'<i>\1</i>', text)
    return text


def parse_md_table(table_text: str):
    """
    Parse a Markdown table string into a list-of-lists (rows × cols).
    Returns None if the input doesn't look like a valid table.
    """
    lines = [l.strip() for l in table_text.strip().splitlines() if l.strip()]
    # Must have at least header + separator + one data row
    if len(lines) < 3:
        return None
    # Filter out separator row (only dashes, pipes, colons)
    data_lines = [
        l for l in lines
        if not re.fullmatch(r'[\|\-:\s]+', l)
    ]
    if not data_lines:
        return None
    rows = []
    for line in data_lines:
        # Strip leading/trailing pipe, then split on pipe
        cells = [c.strip() for c in line.strip('|').split('|')]
        rows.append(cells)
    return rows


def build_md_table_flowable(rows, styles, page_width=470):
    """
    Turn a parsed Markdown table (list-of-lists) into a ReportLab Table.
    First row is treated as the header.
    """
    if not rows:
        return None

    num_cols = max(len(r) for r in rows)
    # Normalise row lengths
    rows = [r + [''] * (num_cols - len(r)) for r in rows]

    col_w = page_width / num_cols

    tbl_data = []
    for i, row in enumerate(rows):
        style = styles['TableHeader'] if i == 0 else styles['TableCell']
        tbl_data.append([
            Paragraph(md_inline_to_xml(escape_xml(cell)), style)
            for cell in row
        ])

    tbl = Table(tbl_data, colWidths=[col_w] * num_cols, repeatRows=1)
    tbl.setStyle(TableStyle([
        # Header row
        ('BACKGROUND', (0, 0), (-1, 0), colors.HexColor('#1e3a5f')),
        ('TEXTCOLOR',  (0, 0), (-1, 0), colors.white),
        # Alternating body rows
        ('ROWBACKGROUNDS', (0, 1), (-1, -1),
         [colors.HexColor('#f0f4f8'), colors.white]),
        # Grid
        ('GRID',       (0, 0), (-1, -1), 0.5, colors.HexColor('#cccccc')),
        # Alignment & padding
        ('VALIGN',     (0, 0), (-1, -1), 'TOP'),
        ('TOPPADDING', (0, 0), (-1, -1), 5),
        ('BOTTOMPADDING', (0, 0), (-1, -1), 5),
        ('LEFTPADDING',  (0, 0), (-1, -1), 6),
        ('RIGHTPADDING', (0, 0), (-1, -1), 6),
    ]))
    return tbl


def content_to_flowables(content: str, styles: dict, page_width=470):
    """
    Convert a multi-line LLM output string (may contain Markdown)
    into a list of ReportLab flowables.

    Handles:
    - ### Sub-headings
    - **Bold** / *italic* inline
    - Bullet lines starting with  •  or  -
    - Numbered lines  1. 2. 3.
    - Markdown tables  (| col | col |)
    - Plain paragraphs (blank-line delimited)
    """
    flowables = []

    # Split into logical blocks separated by blank lines
    blocks = re.split(r'\n\s*\n', content.strip())

    for block in blocks:
        block = block.strip()
        if not block:
            continue

        # ── Markdown table block ──────────────────────────────────────
        lines_in_block = block.splitlines()
        has_pipe = any('|' in l for l in lines_in_block)
        has_sep  = any(re.fullmatch(r'[\|\-:\s]+', l.strip()) for l in lines_in_block)

        if has_pipe and has_sep:
            rows = parse_md_table(block)
            if rows:
                tbl = build_md_table_flowable(rows, styles, page_width)
                if tbl:
                    flowables.append(Spacer(1, 4))
                    flowables.append(tbl)
                    flowables.append(Spacer(1, 6))
                    continue

        # ── Line-by-line rendering for everything else ─────────────
        for line in lines_in_block:
            line = line.strip()
            if not line:
                flowables.append(Spacer(1, 4))
                continue

            # ### Sub-heading
            if line.startswith('### '):
                text = escape_xml(line[4:].strip())
                text = md_inline_to_xml(text)
                flowables.append(Paragraph(text, styles['SubHeading']))

            # ## Section heading inside content (rare but possible)
            elif line.startswith('## '):
                text = escape_xml(line[3:].strip())
                text = md_inline_to_xml(text)
                flowables.append(Paragraph(text, styles['SectionHeading']))

            # Bullet point:  • item  or  - item  or  * item
            elif re.match(r'^[•\-\*]\s+', line):
                text = re.sub(r'^[•\-\*]\s+', '', line)
                text = escape_xml(text)
                text = md_inline_to_xml(text)
                flowables.append(Paragraph('• ' + text, styles['BulletBody']))

            # Numbered item:  1. item
            elif re.match(r'^\d+\.\s+', line):
                text = escape_xml(line)
                text = md_inline_to_xml(text)
                flowables.append(Paragraph(text, styles['BulletBody']))

            # Plain text / inline bold+italic
            else:
                text = escape_xml(line)
                text = md_inline_to_xml(text)
                flowables.append(Paragraph(text, styles['Body']))

    return flowables


# ─────────────────────────────────────────────────────────────────────
# Metadata table
# ─────────────────────────────────────────────────────────────────────

def build_meta_table(report_date, tickers, macro, period, styles):
    data = [
        [Paragraph('<b>Generated</b>', styles['TableCell']),
         Paragraph(escape_xml(report_date), styles['TableCell'])],
        [Paragraph('<b>Tickers</b>', styles['TableCell']),
         Paragraph(escape_xml(', '.join(tickers)), styles['TableCell'])],
        [Paragraph('<b>Macro</b>', styles['TableCell']),
         Paragraph(escape_xml(', '.join(macro)), styles['TableCell'])],
        [Paragraph('<b>Period</b>', styles['TableCell']),
         Paragraph(escape_xml(period), styles['TableCell'])],
    ]
    tbl = Table(data, colWidths=[100, 370])
    tbl.setStyle(TableStyle([
        ('BACKGROUND', (0, 0), (-1, -1), colors.HexColor('#eef2f7')),
        ('GRID',       (0, 0), (-1, -1), 0.5, colors.HexColor('#cccccc')),
        ('VALIGN',     (0, 0), (-1, -1), 'MIDDLE'),
        ('TOPPADDING', (0, 0), (-1, -1), 5),
        ('BOTTOMPADDING', (0, 0), (-1, -1), 5),
        ('LEFTPADDING',  (0, 0), (-1, -1), 8),
    ]))
    return tbl


# ─────────────────────────────────────────────────────────────────────
# Page numbering
# ─────────────────────────────────────────────────────────────────────

def add_page_number(canvas, doc):
    canvas.saveState()
    canvas.setFont('Helvetica', 8)
    canvas.setFillColor(colors.HexColor('#888888'))
    page_num = f'Page {doc.page}'
    canvas.drawCentredString(letter[0] / 2, 0.4 * inch, page_num)
    canvas.restoreState()


# ─────────────────────────────────────────────────────────────────────
# Main builder
# ─────────────────────────────────────────────────────────────────────

def build_pdf_report(
    report_date, tickers, macro, period,
    trend_report, anomaly_report, risk_report,
    fundamental_report, macro_report,
    output_path,
):
    Path(output_path).parent.mkdir(parents=True, exist_ok=True)

    doc = SimpleDocTemplate(
        output_path,
        pagesize=letter,
        rightMargin=0.75 * inch,
        leftMargin=0.75 * inch,
        topMargin=0.85 * inch,
        bottomMargin=0.75 * inch,
    )

    styles = build_styles()
    story  = []

    # ── Cover / header ─────────────────────────────────────────────
    story.append(Paragraph('FinAgent AI Analysis Report', styles['Title']))
    story.append(Spacer(1, 4))
    story.append(HRFlowable(width='100%', thickness=2,
                            color=colors.HexColor('#1e3a5f'), spaceAfter=8))
    story.append(build_meta_table(report_date, tickers, macro, period, styles))
    story.append(Spacer(1, 16))

    # ── Sections ───────────────────────────────────────────────────
    sections = [
        ('1. Market Trend Summary',           trend_report),
        ('2. Anomaly Detection',               anomaly_report),
        ('3. Risk Commentary & Asset Comparison', risk_report),
        ('4. Fundamental Valuation Analysis', fundamental_report),
        ('5. Macro Environment Analysis',     macro_report),
    ]

    for title_text, content in sections:
        # Section heading
        story.append(Paragraph(escape_xml(title_text), styles['SectionHeading']))
        story.append(HRFlowable(width='100%', thickness=0.5,
                                color=colors.HexColor('#b0c4de'), spaceAfter=6))

        if not content or content.strip() in ('N/A', ''):
            story.append(Paragraph('No data available.', styles['Body']))
        else:
            story.extend(content_to_flowables(str(content), styles))

        story.append(Spacer(1, 10))

    # ── Footer note ────────────────────────────────────────────────
    story.append(HRFlowable(width='100%', thickness=0.5,
                            color=colors.HexColor('#cccccc'), spaceAfter=6))
    story.append(Paragraph(
        'Generated by FinAgent · Groq Llama 3.3 70B · '
        'Yahoo Finance / FRED / alternative.me',
        styles['Footer'],
    ))

    doc.build(story, onFirstPage=add_page_number, onLaterPages=add_page_number)
    return output_path


# ─────────────────────────────────────────────────────────────────────
# Run
# ─────────────────────────────────────────────────────────────────────

report_date = date.today().strftime('%Y-%m-%d')
pdf_path    = f'outputs/reports/ai_analysis_{report_date}.pdf'

pdf_out = build_pdf_report(
    report_date      = report_date,
    tickers          = TICKERS,
    macro            = MACRO,
    period           = PERIOD,
    trend_report     = globals().get('trend_report',       'N/A'),
    anomaly_report   = globals().get('anomaly_report',     'N/A'),
    risk_report      = globals().get('risk_report',        'N/A'),
    fundamental_report = globals().get('fundamental_report','N/A'),
    macro_report     = globals().get('macro_report',       'N/A'),
    output_path      = pdf_path,
)

print(f'✅ PDF report saved: {pdf_out}')


✅ PDF report saved: outputs/reports/ai_analysis_2026-05-21.pdf


In [14]:
# ── Tổng kết pipeline ───────────────────────────────────────────────────
print('Generated files:')
for folder in ['data/raw', 'data/processed', 'outputs/charts', 'outputs/reports']:
    files = sorted(Path(folder).glob('*'))
    if files:
        print(f'\n  📁 {folder}/')
        for f in files:
            size = f.stat().st_size
            size_str = f'{size/1024:.1f} KB' if size > 1024 else f'{size} B'
            print(f'     {f.name:45} {size_str}')

Generated files:

  📁 data/raw/
     AAPL_fundamentals_av.json                     515 B
     AAPL_prices.csv                               6.2 KB
     CL_F_prices.csv                               4.9 KB
     finagent_config.json                          453 B
     GC_F_prices.csv                               4.3 KB
     GOOG_prices.csv                               6.2 KB
     MSFT_fundamentals_av.json                     527 B
     MSFT_prices.csv                               6.1 KB
     news_data.json                                72.2 KB
     NVDA_fundamentals_av.json                     521 B
     NVDA_prices.csv                               6.3 KB

  📁 data/processed/
     AAPL_clean.csv                                18.4 KB
     CL_F_clean.csv                                14.8 KB
     fear_greed.csv                                7.3 KB
     fundamentals.csv                              634 B
     GC_F_clean.csv                                14.3 KB
     GOOG_clean.csv 